In [2]:
# ============================================
# FEATURE ENGINEERING
# STEP 1: LOAD MERGED DATASET
# ============================================

import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

merged_path = processed_path / "merged_dataset.csv"

print("Looking for:")
print(merged_path)

if not merged_path.exists():
    print("\n❌ merged_dataset.csv not found!")
else:
    df = pd.read_csv(
        merged_path,
        parse_dates=["timestamp"]
    )

    print("\n✅ Merged dataset loaded successfully!")
    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    display(df.head())

Looking for:
../data/processed/merged_dataset.csv

✅ Merged dataset loaded successfully!
Shape: (65535, 19)

Columns:
['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'finger_stick_value', 'sleep_duration_hours', 'sleep_quality', 'time_diff_minutes', 'target_glucose_30min', 'target_timestamp_30min', 'target_time_diff_minutes']

First 5 rows:


,Patient,timestamp,glucose_value,meal_type,meal_carbs,bolus_type,bolus_dose,basal_value,temp_basal_value,exercise_type,exercise_duration,exercise_intensity,finger_stick_value,sleep_duration_hours,sleep_quality,time_diff_minutes,target_glucose_30min,target_timestamp_30min,target_time_diff_minutes
0,540-ws-training,2027-05-19 11:36:29,76,NaN,NaN,normal,0.8,0.95,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,71.0,2027-05-19 12:06:29,30.0
1,540-ws-training,2027-05-19 11:41:29,72,NaN,NaN,normal,0.8,0.95,NaN,NaN,NaN,NaN,80.0,NaN,NaN,5.0,78.0,2027-05-19 12:11:29,30.0
2,540-ws-training,2027-05-19 11:46:29,68,NaN,NaN,normal,0.8,0.95,NaN,NaN,NaN,NaN,80.0,NaN,NaN,5.0,90.0,2027-05-19 12:16:29,30.0
3,540-ws-training,2027-05-19 11:51:29,65,NaN,NaN,normal,0.8,0.95,NaN,NaN,NaN,NaN,80.0,NaN,NaN,5.0,99.0,2027-05-19 12:21:29,30.0
4,540-ws-training,2027-05-19 11:56:29,63,NaN,NaN,normal,0.8,0.95,NaN,NaN,NaN,NaN,80.0,NaN,NaN,5.0,110.0,2027-05-19 12:26:29,30.0


In [3]:
# ============================================
# STEP 2: GLUCOSE HISTORY FEATURES
# ============================================

# Make sure data is sorted correctly
df = df.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

# Previous glucose readings
df["glucose_lag_1"] = (
    df.groupby("Patient")["glucose_value"]
    .shift(1)
)

df["glucose_lag_2"] = (
    df.groupby("Patient")["glucose_value"]
    .shift(2)
)

df["glucose_lag_3"] = (
    df.groupby("Patient")["glucose_value"]
    .shift(3)
)

df["glucose_lag_6"] = (
    df.groupby("Patient")["glucose_value"]
    .shift(6)
)

# Glucose change over recent readings
df["glucose_change_5min"] = (
    df["glucose_value"] -
    df["glucose_lag_1"]
)

df["glucose_change_15min"] = (
    df["glucose_value"] -
    df["glucose_lag_3"]
)

print("✅ Glucose history features created!")

print("\nNew columns:")
print([
    "glucose_lag_1",
    "glucose_lag_2",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_change_5min",
    "glucose_change_15min"
])

print("\nPreview:")
display(
    df[
        [
            "Patient",
            "timestamp",
            "glucose_value",
            "glucose_lag_1",
            "glucose_lag_2",
            "glucose_lag_3",
            "glucose_lag_6",
            "glucose_change_5min",
            "glucose_change_15min"
        ]
    ].head(10)
)

✅ Glucose history features created!

New columns:
['glucose_lag_1', 'glucose_lag_2', 'glucose_lag_3', 'glucose_lag_6', 'glucose_change_5min', 'glucose_change_15min']

Preview:


,Patient,timestamp,glucose_value,glucose_lag_1,glucose_lag_2,glucose_lag_3,glucose_lag_6,glucose_change_5min,glucose_change_15min
0,540-ws-training,2027-05-19 11:36:29,76,NaN,NaN,NaN,NaN,NaN,NaN
1,540-ws-training,2027-05-19 11:41:29,72,76.0,NaN,NaN,NaN,-4.0,NaN
2,540-ws-training,2027-05-19 11:46:29,68,72.0,76.0,NaN,NaN,-4.0,NaN
3,540-ws-training,2027-05-19 11:51:29,65,68.0,72.0,76.0,NaN,-3.0,-11.0
4,540-ws-training,2027-05-19 11:56:29,63,65.0,68.0,72.0,NaN,-2.0,-9.0
5,540-ws-training,2027-05-19 12:01:29,66,63.0,65.0,68.0,NaN,3.0,-2.0
6,540-ws-training,2027-05-19 12:06:29,71,66.0,63.0,65.0,76.0,5.0,6.0
7,540-ws-training,2027-05-19 12:11:29,78,71.0,66.0,63.0,72.0,7.0,15.0
8,540-ws-training,2027-05-19 12:16:29,90,78.0,71.0,66.0,68.0,12.0,24.0
9,540-ws-training,2027-05-19 12:21:29,99,90.0,78.0,71.0,65.0,9.0,28.0


In [4]:
# ============================================
# STEP 3: TIME FEATURES
# ============================================

df["hour"] = df["timestamp"].dt.hour

df["day_of_week"] = (
    df["timestamp"].dt.dayofweek
)

print("✅ Time features created!")

print("\nTime feature statistics:")

display(
    df[
        [
            "timestamp",
            "hour",
            "day_of_week"
        ]
    ].head(10)
)

✅ Time features created!

Time feature statistics:


,timestamp,hour,day_of_week
0,2027-05-19 11:36:29,11,2
1,2027-05-19 11:41:29,11,2
2,2027-05-19 11:46:29,11,2
3,2027-05-19 11:51:29,11,2
4,2027-05-19 11:56:29,11,2
5,2027-05-19 12:01:29,12,2
6,2027-05-19 12:06:29,12,2
7,2027-05-19 12:11:29,12,2
8,2027-05-19 12:16:29,12,2
9,2027-05-19 12:21:29,12,2


In [5]:
# ============================================
# STEP 4: SELECT ML FEATURES
# ============================================

feature_columns = [
    "glucose_value",
    "glucose_lag_1",
    "glucose_lag_2",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_change_5min",
    "glucose_change_15min",
    "meal_carbs",
    "bolus_dose",
    "basal_value",
    "temp_basal_value",
    "exercise_duration",
    "exercise_intensity",
    "sleep_duration_hours",
    "sleep_quality",
    "finger_stick_value",
    "hour",
    "day_of_week"
]

target_column = "target_glucose_30min"

print("Number of features:", len(feature_columns))

print("\nFeatures:")
for i, feature in enumerate(feature_columns, 1):
    print(f"{i}. {feature}")

print("\nTarget:")
print(target_column)

Number of features: 18

Features:
1. glucose_value
2. glucose_lag_1
3. glucose_lag_2
4. glucose_lag_3
5. glucose_lag_6
6. glucose_change_5min
7. glucose_change_15min
8. meal_carbs
9. bolus_dose
10. basal_value
11. temp_basal_value
12. exercise_duration
13. exercise_intensity
14. sleep_duration_hours
15. sleep_quality
16. finger_stick_value
17. hour
18. day_of_week

Target:
target_glucose_30min


In [6]:
# ============================================
# STEP 5: CHECK MISSING VALUES
# ============================================

print("Missing values in ML features:")
print(
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing target values:")
print(
    df[target_column].isna().sum()
)

print("\nTotal missing feature values:")
print(
    df[feature_columns]
    .isna()
    .sum()
    .sum()
)

Missing values in ML features:
exercise_duration       35807
exercise_intensity      35807
sleep_duration_hours    13579
sleep_quality           13579
temp_basal_value         6926
meal_carbs               1509
bolus_dose                241
finger_stick_value        179
glucose_lag_6              36
glucose_lag_3              18
glucose_change_15min       18
glucose_lag_2              12
glucose_change_5min         6
glucose_lag_1               6
glucose_value               0
hour                        0
basal_value                 0
day_of_week                 0
dtype: int64

Missing target values:
36

Total missing feature values:
107723


In [7]:
# ============================================
# STEP 6: HANDLE MISSING VALUES
# ============================================

# --------------------------------------------
# 1. Event-based features
# --------------------------------------------

event_features = [
    "meal_carbs",
    "bolus_dose",
    "temp_basal_value",
    "exercise_duration",
    "exercise_intensity"
]

for column in event_features:
    df[column] = df[column].fillna(0)


# --------------------------------------------
# 2. Sleep features
# --------------------------------------------

sleep_features = [
    "sleep_duration_hours",
    "sleep_quality"
]

for column in sleep_features:

    df[column] = (
        df.groupby("Patient")[column]
        .ffill()
    )

    df[column] = df[column].fillna(0)


# --------------------------------------------
# 3. Finger-stick glucose
# --------------------------------------------

df["finger_stick_value"] = (
    df.groupby("Patient")["finger_stick_value"]
    .ffill()
)

# If no previous finger-stick exists,
# use current CGM glucose as fallback
df["finger_stick_value"] = (
    df["finger_stick_value"]
    .fillna(df["glucose_value"])
)


# --------------------------------------------
# 4. Glucose history features
# --------------------------------------------

glucose_history_features = [
    "glucose_lag_1",
    "glucose_lag_2",
    "glucose_lag_3",
    "glucose_lag_6",
    "glucose_change_5min",
    "glucose_change_15min"
]

for column in glucose_history_features:
    df[column] = df[column].fillna(0)


# --------------------------------------------
# 5. Remove rows without target
# --------------------------------------------

before_rows = len(df)

df = df.dropna(
    subset=[target_column]
).copy()

after_rows = len(df)

print("Rows before removing missing target:", before_rows)
print("Rows after removing missing target:", after_rows)
print("Rows removed:", before_rows - after_rows)


# --------------------------------------------
# 6. Final missing-value check
# --------------------------------------------

missing_values = (
    df[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values after processing:")
print(missing_values)

print(
    "\nTotal remaining missing feature values:",
    missing_values.sum()
)

Rows before removing missing target: 65535
Rows after removing missing target: 65499
Rows removed: 36

Missing values after processing:
glucose_value           0
glucose_lag_1           0
hour                    0
finger_stick_value      0
sleep_quality           0
sleep_duration_hours    0
exercise_intensity      0
exercise_duration       0
temp_basal_value        0
basal_value             0
bolus_dose              0
meal_carbs              0
glucose_change_15min    0
glucose_change_5min     0
glucose_lag_6           0
glucose_lag_3           0
glucose_lag_2           0
day_of_week             0
dtype: int64

Total remaining missing feature values: 0


In [8]:
# ============================================
# STEP 7: CREATE FINAL ML DATASET
# ============================================

ml_columns = [
    "Patient",
    "timestamp"
] + feature_columns + [
    target_column
]

ml_df = df[ml_columns].copy()

print("✅ Final ML dataset created!")

print("\nShape:")
print(ml_df.shape)

print("\nNumber of features:", len(feature_columns))

print("\nFeature columns:")
print(feature_columns)

print("\nTarget:")
print(target_column)

print("\nFirst 5 rows:")
display(ml_df.head())

✅ Final ML dataset created!

Shape:
(65499, 21)

Number of features: 18

Feature columns:
['glucose_value', 'glucose_lag_1', 'glucose_lag_2', 'glucose_lag_3', 'glucose_lag_6', 'glucose_change_5min', 'glucose_change_15min', 'meal_carbs', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_duration', 'exercise_intensity', 'sleep_duration_hours', 'sleep_quality', 'finger_stick_value', 'hour', 'day_of_week']

Target:
target_glucose_30min

First 5 rows:


,Patient,timestamp,glucose_value,glucose_lag_1,glucose_lag_2,glucose_lag_3,glucose_lag_6,glucose_change_5min,glucose_change_15min,meal_carbs,...,basal_value,temp_basal_value,exercise_duration,exercise_intensity,sleep_duration_hours,sleep_quality,finger_stick_value,hour,day_of_week,target_glucose_30min
0,540-ws-training,2027-05-19 11:36:29,76,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.95,0.0,0.0,0.0,0.0,0.0,80.0,11,2,71.0
1,540-ws-training,2027-05-19 11:41:29,72,76.0,0.0,0.0,0.0,-4.0,0.0,0.0,...,0.95,0.0,0.0,0.0,0.0,0.0,80.0,11,2,78.0
2,540-ws-training,2027-05-19 11:46:29,68,72.0,76.0,0.0,0.0,-4.0,0.0,0.0,...,0.95,0.0,0.0,0.0,0.0,0.0,80.0,11,2,90.0
3,540-ws-training,2027-05-19 11:51:29,65,68.0,72.0,76.0,0.0,-3.0,-11.0,0.0,...,0.95,0.0,0.0,0.0,0.0,0.0,80.0,11,2,99.0
4,540-ws-training,2027-05-19 11:56:29,63,65.0,68.0,72.0,0.0,-2.0,-9.0,0.0,...,0.95,0.0,0.0,0.0,0.0,0.0,80.0,11,2,110.0


In [9]:
# ============================================
# STEP 8: VALIDATE FINAL ML DATASET
# ============================================

print("========== ML DATASET VALIDATION ==========")

print("\nShape:")
print(ml_df.shape)

print("\nMissing values:")
print(
    ml_df.isna().sum()
)

print("\nDuplicate Patient + timestamp records:")

duplicates = ml_df.duplicated(
    subset=["Patient", "timestamp"]
).sum()

print(duplicates)

print("\nPatients:")
print(
    ml_df["Patient"].nunique()
)

print(
    ml_df["Patient"].unique()
)

print("\nDate range:")

print(
    "Start:",
    ml_df["timestamp"].min()
)

print(
    "End:",
    ml_df["timestamp"].max()
)

print("\nTarget statistics:")
print(
    ml_df[target_column].describe()
)

========== ML DATASET VALIDATION ==========

Shape:
(65499, 21)

Missing values:
Patient                 0
timestamp               0
glucose_value           0
glucose_lag_1           0
glucose_lag_2           0
glucose_lag_3           0
glucose_lag_6           0
glucose_change_5min     0
glucose_change_15min    0
meal_carbs              0
bolus_dose              0
basal_value             0
temp_basal_value        0
exercise_duration       0
exercise_intensity      0
sleep_duration_hours    0
sleep_quality           0
finger_stick_value      0
hour                    0
day_of_week             0
target_glucose_30min    0
dtype: int64

Duplicate Patient + timestamp records:
0

Patients:
6
<StringArray>
['540-ws-training', '544-ws-training', '552-ws-training', '567-ws-training',
 '584-ws-training', '596-ws-training']
Length: 6, dtype: str

Date range:
Start: 2025-04-16 11:17:05
End: 2027-07-03 23:26:44

Target statistics:
count    65499.000000
mean       157.736194
std         60.874881
mi

In [10]:
# ============================================
# STEP 9: SAVE FEATURE-ENGINEERED DATASET
# ============================================

output_path = processed_path / "ml_dataset.csv"

ml_df.to_csv(
    output_path,
    index=False
)

print("✅ Feature-engineered ML dataset saved!")
print("Path:", output_path)
print("Shape:", ml_df.shape)

✅ Feature-engineered ML dataset saved!
Path: ../data/processed/ml_dataset.csv
Shape: (65499, 21)
